# DP8 集計（CPU）— A・B をまとめて、事前に決めたラベルを出す

ゲート：`notes/dp8_rarity_bridge_gate.md`。**GPU も Claude の API も使いません（費用 $0）**。

- `DP8_GEMMA_A` と `DP8_GEMMA_B` の**両方が監査（セル8）まで終わってから**回してね（終わっていなければ、何も書かずに止まります）。
- Drive の記録を読み、集計を 2 回計算してバイト単位で一致することを確かめます。そのあと、監査をまとめてラベルを出します。
- 結果は `DECISION_POINTS/dp8/` に **数とラベルだけ** を書きます。問題の中身は表示しません。

**使い方**：CPU のランタイム（ハイメモリは不要）。シークレットは `GITHUB_TOKEN`。「すべてのセルを実行」で 5〜15 分。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。
- **エラーのとき**：通知の本文にエラーの文面（トレースバック）が入ります。全文は Drive の `DECISION_POINTS/_errors/` にも保存されるので、Claude が直接読めます。

In [ ]:
# Cell 1 — Drive, clone this head, read both GPU parts (units + audits), analyse twice, labels, write, summary.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。エラーのときはエラーの文面も送ります（鍵・トークンらしい文字は伏せ字。全文は Drive の DECISION_POINTS/_errors）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'DP8 集計（CPU、A・B をまとめてラベルを出す）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'GodPork - DP8 analyze', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
import os, re as _re, traceback as _tb, datetime as _edt
ERR_SLUG = 'DP8_ANALYZE'   # Drive のエラーファイルの名前の頭
_SECRET_RE = _re.compile(r"sk-ant-[A-Za-z0-9_\-]+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9]{8,}|hf_[A-Za-z0-9]{8,}|x-access-token:[^@\s]+@")
def _scrub(s):
    s = _SECRET_RE.sub('[伏せ字]', s)
    return '\n'.join(l if len(l) <= 300 else l[:300] + ' …（長いので省略）' for l in s.splitlines())
def _error_text(result):
    e = result.error_in_exec if result.error_in_exec is not None else result.error_before_exec
    tb = ''.join(_tb.format_exception(type(e), e, e.__traceback__))
    first = (getattr(getattr(result, 'info', None), 'raw_cell', '') or '').strip().splitlines()[:1]
    return _scrub('セル: ' + (first[0][:100] if first else '?') + '\n' + tb)
def _save_error(text):
    base = '/content/drive/MyDrive'
    if not os.path.isdir(base):
        return None
    d = os.path.join(base, 'DECISION_POINTS', '_errors')
    os.makedirs(d, exist_ok=True)
    stem = ERR_SLUG + '_' + _edt.datetime.now(_edt.timezone(_edt.timedelta(hours=9))).strftime('%Y%m%d_%H%M%S')
    name, n = stem + '.txt', 1
    while os.path.exists(os.path.join(d, name)):
        n += 1
        name = stem + '_' + str(n) + '.txt'
    with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
        fh.write('[' + NB_LABEL + ']\n' + text)
    return 'DECISION_POINTS/_errors/' + name
def _short(text, limit=3300):
    b = text.encode('utf-8')
    if len(b) <= limit:
        return text
    return '…（前のほうは省略。全文は Drive）\n' + b[-limit:].decode('utf-8', 'ignore')
def _on_error(result):
    if result.error_in_exec is None and result.error_before_exec is None:
        return
    try:
        text = _error_text(result)
    except Exception as e2:
        text = 'エラーの文面を作れませんでした：' + type(e2).__name__
    try:
        where = _save_error(text)
    except Exception as e3:
        where = None
        text += '\n（Drive に保存できませんでした：' + type(e3).__name__ + '）'
    print('エラーの全文を通知に送ります' + ('（Drive：' + where + '）' if where else ''))
    notify('止まりました：エラーが出ています。' + ('全文は Drive の ' + where if where else 'Drive には保存できませんでした')
           + '\n\n' + _short(text), tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

drive.mount("/content/drive")

EXPECTED_HEAD = "88b029da398556fc3b3c49903a318d7d5e81ba50"
RUN_HEAD = "88b029da398556fc3b3c49903a318d7d5e81ba50"
BRANCH = "decision-points"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")
DP_ROOT = Path("/content/drive/MyDrive/DECISION_POINTS/dp8")
PARTS = ("A", "B")
def run_base(h):
    return DP_ROOT / ("DP8_GEMMA_" + h + "_20261015")
for h in PARTS:
    b = run_base(h)
    if not ((b / "RUN_IDENTITY.json").exists() and (b / f"AUDIT_{h}.json").exists()):
        raise SystemExit("A・B の 2 本とも、監査（セル8）まで終わってから実行してね（" + h + " がまだ）。")
    assert not any(b.glob("RUN_FAILED*.txt")), (h, "has a failure marker — show Claude")

try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
assert token and str(token).strip(), "Colab のシークレットに GITHUB_TOKEN を登録してね（ノートブックへのアクセスもオン）。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=str(token).strip(), GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
token = None
os.chdir('/content')   # セル1 をやり直しても、消すフォルダの中にいないように（2026-10-14）
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
env.pop("GITHUB_TOKEN", None)
subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "--detach", EXPECTED_HEAD], check=True)
assert subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() == EXPECTED_HEAD
# the analysis code is the code the GPU parts ran (no change to any pork module between RUN_HEAD and this head)
assert subprocess.run(["git", "-C", str(REPO), "merge-base", "--is-ancestor", RUN_HEAD, EXPECTED_HEAD]).returncode == 0
assert subprocess.run(["git", "-C", str(REPO), "diff", "--quiet", RUN_HEAD, EXPECTED_HEAD, "--", "pork",
                       "notes/dp8_rarity_bridge_gate.md"]).returncode == 0, "pork/ or the gate changed after the run"
sys.path.insert(0, str(REPO))
from pork import dp8_bridge as d8
from pork import dp8_runner as rr
assert d8.gate_doc_sha256(REPO) == d8.GATE_DOC_SHA256, "DP8 gate document differs from the frozen hash"

rows_raw, DATA_ID = d8.load_rows_from_hub()
ITEMS = d8.select_items(rows_raw)
d8.check_cr1_candidates(rows_raw); del rows_raw
ITEMS_SHA = d8.items_sha256(ITEMS)
TEMPLATES = [it["template"] for it in ITEMS]
cases, audits, commits, idents = [], {}, {}, {}
for h in PARTS:
    b = run_base(h)
    ident = json.loads((b / "RUN_IDENTITY.json").read_text())
    assert ident["items_sha256"] == ITEMS_SHA and ident["half"] == h and ident["gate_doc_sha256"] == d8.GATE_DOC_SHA256
    assert ident["code_commit"] == RUN_HEAD, (h, "was run at a different code commit", ident["code_commit"])
    assert ident["dataset_revision"] == DATA_ID["dataset_revision"]
    cases += rr.load_cases(d8.halves()[h], units_dir=b / "units", code_commit=ident["code_commit"], items_hash=ITEMS_SHA)
    audits[h] = json.loads((b / f"AUDIT_{h}.json").read_text())
    assert audits[h]["code_commit"] == ident["code_commit"]
    commits[h] = ident["code_commit"]; idents[h] = ident
for key in ("tau_excluded_ids", "eos_ids", "model_revision", "dataset_revision", "vocab", "items_sha256"):
    assert idents["A"][key] == idents["B"][key], ("the two parts differ in", key)
for h in PARTS:
    b = run_base(h)
    par = json.loads((b / "SAMPLER_PARITY.json").read_text())
    e2 = json.loads((b / "E2.json").read_text()); g1 = json.loads((b / "G1.json").read_text())
    assert par["pass"] is True and all(r["argmax_agree"] for r in e2) and all(r["identical"] for r in g1), (h, "pre-checks")
cases = sorted(cases, key=lambda c: c["row"])
assert [c["row"] for c in cases] == list(range(d8.N_ITEMS))
assert [c["template"] for c in cases] == TEMPLATES
r1 = json.dumps(d8.analyze(cases), sort_keys=True).encode()
r2 = json.dumps(d8.analyze(cases), sort_keys=True).encode()
assert r1 == r2, "analysis not reproducible"
RES = json.loads(r1)
import numpy as np, scipy
result = {"schema": d8.SCHEMA, "analysis": RES, "items_sha256": ITEMS_SHA, "code_commits": commits,
          "gate_doc_sha256": d8.GATE_DOC_SHA256, "analysis_head": EXPECTED_HEAD,
          "versions": {"numpy": np.__version__, "scipy": scipy.__version__}}
result["result_sha256"] = d8.result_sha(result)
audit = rr.combine_audits(audits); audit["per_half"] = audits
labels = d8.final_labels(RES, audit)
final = {"final_labels": labels, "guard_ok": RES["guard_ok"], "base_accuracy": RES["base_accuracy"],
         "match_coverage": RES["match_coverage"], "coverage_ok": RES["coverage_ok"],
         "audit_pass": audit["pass"], "audit_diverged_rows": audit["diverged_rows"],
         "audit_references": audit["audit_references"], "result_sha256": result["result_sha256"],
         "audit_sha256": hashlib.sha256(json.dumps(audit, sort_keys=True).encode()).hexdigest(), "code_commits": commits}

def atomic_json(p, value):
    tmp = p.with_name(p.name + ".tmp")
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=1, sort_keys=True) + "\n", encoding="utf-8")
    os.replace(tmp, p)
    assert json.loads(p.read_text(encoding="utf-8")) == value
atomic_json(DP_ROOT / "RESULT_DP8.json", result)
atomic_json(DP_ROOT / "AUDIT_DP8.json", audit)
atomic_json(DP_ROOT / "FINAL_DP8.json", final)

r3 = lambda v: "—" if v is None else f"{v:+.3f}"
ci = lambda r: "[" + ", ".join(r3(v) for v in r["ci"]) + "]"
m1, m2, m3, m4 = (RES[k] for k in ("MAIN1_e2_minus_e_alt", "MAIN2_spearman_r_e_alt", "MAIN3_rare_share_d_s", "MAIN4_e_tau_minus_e2"))
print(f"土台の正解率 {RES['base_accuracy']:.3f}（ガード {'OK' if RES['guard_ok'] else '外れ'}）　位置 {RES['positions_all']}・合う位置 {RES['positions_matched']}（{RES['match_coverage']:.3f}）")
print("主1 e2 − e_alt の平均", r3(m1["mean"]), ci(m1), "→", labels["main1"])
print("主2 (r, e_alt) の問題内スピアマン", r3(m2["mean"]), ci(m2), "→", labels["main2"])
print("主3 r ≤ 0.05 の寄与の割合", r3(m3["point"]), ci(m3), "→", labels["main3"])
print("主4 e_tau − e2 の平均", r3(m4["mean"]), ci(m4), "→", labels["main4"])
print(f"監査: {'合格' if audit['pass'] else '不合格'}（ずれた問題 {audit['diverged_rows']}）")
print("結果 sha256:", result["result_sha256"][:16], "| 保存: DECISION_POINTS/dp8/{RESULT,AUDIT,FINAL}_DP8.json")
print("✅ 集計おわり。Claude に知らせてね")
notify('終わりました：DP8 の集計（主1 ' + labels['main1'] + '・主2 ' + labels['main2'] + '・主3 ' + labels['main3'] + '・主4 ' + labels['main4'] + '）', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()